In [ ]:
"""
=============================================================================
COMPLETE RAG SYSTEM
=============================================================================
Just run this single cell and you're done! No files, no setup needed.
"""

# STEP 1: Install required packages (run once)
print("📦 Installing packages... (this takes ~2 minutes)")
!pip install -q langchain langchain-groq langchain-community langgraph
!pip install -q chromadb sentence-transformers
!pip install -q gradio pypdf python-docx beautifulsoup4
!pip install -q tiktoken pydantic pydantic-settings
print("✅ Installation complete!\n")

# STEP 2: Import all required libraries
import gradio as gr
from typing import TypedDict, Annotated, List, Dict
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from pathlib import Path
import chromadb
from chromadb.config import Settings as ChromaSettings
from sentence_transformers import SentenceTransformer
import pypdf
from docx import Document as DocxDocument
import tiktoken
import tempfile
import os

# STEP 3: Configuration - CHANGE YOUR API KEY HERE!
GROQ_API_KEY = ""  # 👈 PUT YOUR KEY HERE
LLM_MODEL = "groq/compound"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K_RESULTS = 6
CHUNK_SIZE = 900
CHUNK_OVERLAP = 150

# STEP 4: Document Processing Classes
class DocumentLoader:
    """Load documents from various file formats"""

    def load_document(self, file_path: str) -> Dict[str, str]:
        path = Path(file_path)
        suffix = path.suffix.lower()

        if suffix == '.pdf':
            return self._load_pdf(path)
        elif suffix == '.txt':
            return self._load_txt(path)
        elif suffix == '.docx':
            return self._load_docx(path)
        else:
            raise ValueError(f"Unsupported file format: {suffix}")

    def _load_pdf(self, file_path: Path) -> Dict[str, str]:
        text = ""
        with open(file_path, 'rb') as file:
            pdf_reader = pypdf.PdfReader(file)
            for page in pdf_reader.pages:
                text += page.extract_text()
        return {"source": str(file_path.name), "content": text}

    def _load_txt(self, file_path: Path) -> Dict[str, str]:
        with open(file_path, 'r', encoding='utf-8') as file:
            text = file.read()
        return {"source": str(file_path.name), "content": text}

    def _load_docx(self, file_path: Path) -> Dict[str, str]:
        doc = DocxDocument(file_path)
        text = "\n".join([para.text for para in doc.paragraphs])
        return {"source": str(file_path.name), "content": text}


class TextSplitter:
    """Split text into chunks for embedding"""

    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 50):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.encoding = tiktoken.get_encoding("cl100k_base")

    def split_text(self, document: Dict[str, str]) -> List[Dict[str, str]]:
        text = document["content"]
        source = document["source"]

        tokens = self.encoding.encode(text)
        chunks = []

        for i in range(0, len(tokens), self.chunk_size - self.chunk_overlap):
            chunk_tokens = tokens[i:i + self.chunk_size]
            chunk_text = self.encoding.decode(chunk_tokens)

            chunks.append({
                "content": chunk_text,
                "source": source,
                "chunk_id": len(chunks)
            })

        return chunks


# STEP 5: Vector Store Class
class VectorStore:
    """Manage vector embeddings and similarity search"""

    def __init__(self):
        # Create temporary directory for ChromaDB
        self.persist_dir = tempfile.mkdtemp()
        self.client = chromadb.PersistentClient(
            path=self.persist_dir,
            settings=ChromaSettings(anonymized_telemetry=False)
        )
        self.collection = self.client.get_or_create_collection(name="documents")

        print("🔄 Loading embedding model... (first time takes ~30 seconds)")
        self.embedding_model = SentenceTransformer(EMBEDDING_MODEL)
        print("✅ Embedding model loaded!")

    def add_documents(self, chunks: List[Dict[str, str]]):
        """Add document chunks to the vector store"""
        if not chunks:
            return

        ids = [f"{chunk['source']}_chunk_{chunk['chunk_id']}" for chunk in chunks]
        documents = [chunk['content'] for chunk in chunks]
        metadatas = [{"source": chunk['source'], "chunk_id": chunk['chunk_id']}
                     for chunk in chunks]

        # Generate embeddings
        print(f"🔄 Generating embeddings for {len(chunks)} chunks...")
        embeddings = self.embedding_model.encode(documents).tolist()

        self.collection.add(
            ids=ids,
            embeddings=embeddings,
            documents=documents,
            metadatas=metadatas
        )

        print(f"✅ Added {len(chunks)} chunks to vector store")

    def search(self, query: str, top_k: int = TOP_K_RESULTS) -> List[Dict]:
        """Search for relevant documents"""
        # Generate query embedding
        query_embedding = self.embedding_model.encode([query])[0].tolist()

        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k
        )

        documents = []
        if results['ids'] and results['ids'][0]:
            for i in range(len(results['ids'][0])):
                documents.append({
                    'content': results['documents'][0][i],
                    'metadata': results['metadatas'][0][i],
                    'distance': results['distances'][0][i] if results['distances'] else None
                })

        return documents

    def get_collection_count(self) -> int:
        """Get the number of documents in the collection"""
        return self.collection.count()


# STEP 6: RAG State Definition
class RAGState(TypedDict):
    """State for the RAG graph"""
    messages: Annotated[List[HumanMessage | AIMessage], "Conversation messages"]
    query: str
    retrieved_docs: List[dict]
    context: str
    answer: str
    sources: List[str]


# STEP 7: RAG Graph with LangGraph
class RAGGraph:
    """LangGraph-based RAG system"""

    def __init__(self, vector_store: VectorStore):
        self.llm = ChatGroq(
            temperature=0,
            groq_api_key=GROQ_API_KEY,
            model_name=LLM_MODEL
        )

        self.vector_store = vector_store

        # Define the RAG prompt
        self.rag_prompt = ChatPromptTemplate.from_messages([
            ("system", """
You are an internal policy interpretation engine.

STRICT RULES:
1. Answer ONLY using statements that are explicitly present in the provided context.
2. Do NOT introduce calculations, formulas, assumptions, or interpretations that are not directly stated in the text.
3. If the policy does not specify a method (e.g., day-wise calculation), state that the policy is silent.
4. Prefer process descriptions (accrual, eligibility, conditions) over numerical derivations.
5. If multiple interpretations are possible, choose the most conservative, text-literal interpretation.
6. Do NOT rely on common HR practice or industry norms.

Context:
{context}
"""),("human", "{query}"),
        ])

        # Build the graph
        self.graph = self._build_graph()

    def _build_graph(self) -> StateGraph:
        """Build the RAG workflow graph"""
        workflow = StateGraph(RAGState)

        # Add nodes
        workflow.add_node("retrieve", self._retrieve_documents)
        workflow.add_node("generate", self._generate_answer)

        # Add edges
        workflow.set_entry_point("retrieve")
        workflow.add_edge("retrieve", "generate")
        workflow.add_edge("generate", END)

        return workflow.compile()

    def _retrieve_documents(self, state: RAGState) -> RAGState:
        """Retrieve relevant documents from vector store"""
        query = state['query']

        # Search vector store
        docs = self.vector_store.search(query, top_k=TOP_K_RESULTS)

        # Build context from retrieved documents
        context_parts = []
        sources = []

        for i, doc in enumerate(docs, 1):
            context_parts.append(f"[Document {i}]\n{doc['content']}")
            sources.append(doc['metadata']['source'])

        context = "\n\n".join(context_parts)

        return {
            **state,
            "retrieved_docs": docs,
            "context": context,
            "sources": list(set(sources)),
        }

    def _generate_answer(self, state: RAGState) -> RAGState:
        """Generate answer using LLM"""

        # 🔒 HARD GUARD: no context → no generation
        if not state['context'].strip():
            state['answer'] = "The policy does not contain information relevant to this question."
            state['messages'].append(HumanMessage(content=state['query']))
            state['messages'].append(AIMessage(content=state['answer']))
            return state

        # Format the prompt
        messages = self.rag_prompt.format_messages(
            context=state['context'],
            query=state['query']
        )

        # Get LLM response
        response = self.llm.invoke(messages)

        # Update state
        state['answer'] = response.content
        state['messages'].append(HumanMessage(content=state['query']))
        state['messages'].append(AIMessage(content=response.content))

        return state


    def query(self, question: str) -> dict:
        """Process a query through the RAG pipeline"""
        # Initialize state
        initial_state = {
            "messages": [],
            "query": question,
            "retrieved_docs": [],
            "context": "",
            "answer": "",
            "sources": [],
        }

        # Run the graph
        result = self.graph.invoke(initial_state)

        return {
            "answer": result['answer'],
            "sources": result['sources'],
            "num_docs_retrieved": len(result['retrieved_docs'])
        }


# STEP 8: Initialize Components
print("\n🚀 Initializing RAG System...")
vector_store = VectorStore()
rag_graph = RAGGraph(vector_store)
document_loader = DocumentLoader()
text_splitter = TextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
print("✅ RAG System Ready!\n")


# STEP 9: Gradio Interface Functions
def query_rag(question: str) -> tuple[str, str]:
    """Query the RAG system"""
    if not question.strip():
        return "Please enter a question.", ""

    # Check if there are documents in the vector store
    doc_count = vector_store.get_collection_count()
    if doc_count == 0:
        return ("⚠️ No documents in the knowledge base. Please upload documents first "
                "using the 'Document Management' tab."), ""

    # Query the RAG system
    result = rag_graph.query(question)

    # Format the response
    answer = result['answer']
    sources_text = "\n".join([f"📄 {source}" for source in result['sources']])

    if not sources_text:
        sources_text = "No sources found"

    return answer, sources_text


def upload_documents(files) -> str:
    """Upload and process documents"""
    if not files:
        return "⚠️ No files uploaded"

    try:
        total_chunks = 0
        processed_files = []

        for file in files:
            # Load document
            document = document_loader.load_document(file.name)

            # Split into chunks
            chunks = text_splitter.split_text(document)

            # Add to vector store
            vector_store.add_documents(chunks)

            total_chunks += len(chunks)
            processed_files.append(Path(file.name).name)

        doc_count = vector_store.get_collection_count()

        status = f"""✅ Successfully processed {len(files)} document(s)

📊 Details:
- Files processed: {', '.join(processed_files)}
- Chunks created: {total_chunks}
- Total documents in database: {doc_count}
"""
        return status

    except Exception as e:
        return f"❌ Error processing documents: {str(e)}"


def get_database_stats() -> str:
    """Get current database statistics"""
    doc_count = vector_store.get_collection_count()
    return f"""📊 Knowledge Base Statistics:

- Total document chunks: {doc_count}
- Vector store: ChromaDB
- Embedding model: {EMBEDDING_MODEL}
- LLM model: {LLM_MODEL}
- Retrieval: Top {TOP_K_RESULTS} documents
"""


# STEP 10: Build Gradio Interface
with gr.Blocks(theme='Yntec/HaleyCH_Theme_Orange_Green', title="RAG System") as interface:
    gr.Markdown("""
    # 🤖 Production RAG System
    ### Retrieval-Augmented Generation with LangGraph

    Upload documents (PDF, TXT, DOCX) and ask questions about them!
    """)

    with gr.Tabs():
        # Query Tab
        with gr.Tab("💬 Ask Questions"):
            with gr.Row():
                with gr.Column(scale=2):
                    question_input = gr.Textbox(
                        label="Enter your question",
                        placeholder="What would you like to know about your documents?",
                        lines=3
                    )
                    query_btn = gr.Button("🔍 Ask", variant="primary", size="lg")

            with gr.Row():
                with gr.Column():
                    answer_output = gr.Textbox(
                        label="Answer",
                        lines=10,
                        show_copy_button=True
                    )
                with gr.Column():
                    sources_output = gr.Textbox(
                        label="Sources",
                        lines=10
                    )

            gr.Examples(
                examples=[
                    ["What are the main topics covered in the documents?"],
                    ["Summarize the key findings"],
                    ["What recommendations are mentioned?"],
                ],
                inputs=question_input
            )

        # Document Management Tab
        with gr.Tab("📁 Document Management"):
            gr.Markdown("### Upload Documents")
            gr.Markdown("Supported formats: PDF, TXT, DOCX")

            file_upload = gr.File(
                label="Upload Documents",
                file_count="multiple",
                file_types=[".pdf", ".txt", ".docx"]
            )
            upload_btn = gr.Button("📤 Upload & Process", variant="primary")
            upload_status = gr.Textbox(label="Status", lines=8)

            gr.Markdown("### Database Statistics")
            stats_btn = gr.Button("📊 Refresh Statistics")
            stats_output = gr.Textbox(label="Current Statistics", lines=6)

    # Event handlers
    query_btn.click(
        fn=query_rag,
        inputs=[question_input],
        outputs=[answer_output, sources_output]
    )

    question_input.submit(
        fn=query_rag,
        inputs=[question_input],
        outputs=[answer_output, sources_output]
    )

    upload_btn.click(
        fn=upload_documents,
        inputs=[file_upload],
        outputs=[upload_status]
    )

    stats_btn.click(
        fn=get_database_stats,
        outputs=[stats_output]
    )

    # Load initial stats
    interface.load(fn=get_database_stats, outputs=[stats_output])


# STEP 11: Launch the Application
print("\n" + "="*60)
print("🎉 READY TO LAUNCH!")
print("="*60)
print("\n📝 Quick Guide:")
print("1. Click the link that appears below")
print("2. Go to 'Document Management' tab")
print("3. Upload your PDF/TXT/DOCX files")
print("4. Go to 'Ask Questions' tab")
print("5. Type your question and click 'Ask'")
print("\n" + "="*60 + "\n")

interface.launch(share=True, debug=True)

📦 Installing packages... (this takes ~2 minutes)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.21.0 requires opentelemetry-api<=1.37.0,>=1.37.0, but you have opentelemetry-api 1.39.1 which is incompatible.
google-adk 1.21.0 requires opentelemetry-sdk<=1.37.0,>=1.37.0, but you have opentelemetry-sdk 1.39.1 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.39.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-exporter-otlp-proto-common==1.37.0, b


🚀 Initializing RAG System...
🔄 Loading embedding model... (first time takes ~30 seconds)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded!
✅ RAG System Ready!



/tmp/ipython-input-4082956066.py:389: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme='Yntec/HaleyCH_Theme_Orange_Green', title="RAG System") as interface:


theme_schema%400.0.1.json: 0.00B [00:00, ?B/s]


🎉 READY TO LAUNCH!

📝 Quick Guide:
1. Click the link that appears below
2. Go to 'Document Management' tab
3. Upload your PDF/TXT/DOCX files
4. Go to 'Ask Questions' tab
5. Type your question and click 'Ask'


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c1375a1e110abc4b10.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
